# 03 — Beta Diversity

**Paper title:** Maternal gut microbiome diversity and functional potential in early pregnancy are associated with large-for-gestational-age birth (SweMaMi cohort)

**Purpose:** Assesses community composition using Bray-Curtis, Jaccard and Aitchison (robust CLR) distances. Tests dispersion homogeneity (betadisper) before PERMANOVA (adonis2), with random subsampling to check whether unequal dispersion is driven by group-size imbalance.

**Produces:** Figure 4 (PCoA + PERMANOVA R²)


## Setup


In [ ]:
library(dplyr)
library(tidyverse)
library(readxl)
library(vegan)
library(ggplot2)
library(patchwork)


In [ ]:
# Define base path 
base_path <- "data"

## 1. Compute distance matrices (vegdist)


### TP1

In [ ]:
meta_data_tp1 <- read.csv(file.path(base_path,"meta_data_tp1.csv"))


In [ ]:
meta_data_tp1 <- meta_data_tp1 %>%
  mutate(kit1.faecal_sample.barcode = as.character(kit1.faecal_sample.barcode))

meta_data_tp1$Primipara <- factor(meta_data_tp1$Primipara, levels = c("0","1"))
meta_data_tp1$excess_weight <- factor(meta_data_tp1$excess_weight, levels = c("No","Yes"))
meta_data_tp1$Gest_Diabetes <- factor(meta_data_tp1$Gest_Diabetes,levels = c("0","1"))
meta_data_tp1$Q1_healthydiet <- factor(meta_data_tp1$Q1_healthydiet, levels = c("1","0"))
meta_data_tp1$Q2_healthydiet <- factor(meta_data_tp1$Q2_healthydiet, levels = c("1","0"))
meta_data_tp1$Group <- factor(meta_data_tp1$Group, levels = c("Control","Case"))


In [ ]:
head(meta_data_tp1 )

In [ ]:
all_taxa_1 <- read.table(file.path(base_path,"taxa_tp1.tsv.tsv"),
  sep         = "\t",
  header      = TRUE,
  check.names = FALSE
)
head(all_taxa_1)

In [ ]:
new_row <- c()
tot <- colSums(all_taxa_1)
for(i in tot){
    if(i < 100) {
        new_row <- c(new_row,100 - i)
    } else if( i >=100){
        new_row <- c(new_row,0)
    }
}

new_row <- setNames(new_row, names(all_taxa_1))
new_row2 <- as.data.frame(t(new_row))
rownames(new_row2) <- "Other_taxa"
best_filt_new <- rbind(all_taxa_1,new_row2) 
filt_new_arrange_1 <- as.data.frame(best_filt_new)
dim(filt_new_arrange_1)
head(filt_new_arrange_1)

In [ ]:
for_arrange_1 <- as.data.frame(t(filt_new_arrange_1))
head(for_arrange_1)
dim(for_arrange_1)

In [ ]:
beta_taxa_1 <- filt_new_arrange_1 [,colnames(filt_new_arrange_1) %in% meta_data_tp1$kit1.faecal_sample.barcode]

In [ ]:
for_arrange_1 <- as.data.frame(t(beta_taxa_1))
head(for_arrange_1)
dim(for_arrange_1)

In [ ]:
order_taxa_1 <- for_arrange_1[meta_data_tp1$kit1.faecal_sample.barcode, , drop = FALSE]
head(order_taxa_1)

In [ ]:
identical(rownames(order_taxa_1), meta_data_tp1$kit1.faecal_sample.barcode)

In [ ]:
taxa_info <- order_taxa_1
jaccard <- taxa_info %>%                           
  decostand(method = "pa") %>%          
  vegan::vegdist(method = "jaccard")


In [ ]:
aitchison <- order_taxa_1 %>%
  as.matrix() %>%
  vegdist(method = "robust.aitchison")
head(as.matrix(aitchison))

aitchison11 <- as.matrix(aitchison)

In [ ]:
bray_curtis <- vegan::vegdist(order_taxa_1, method = "bray")
head(bray_curtis)

### TP2

In [ ]:
all_taxa_2 <- read.table(file.path(base_path,"taxa_tp2.tsv"),
  sep         = "\t",
  header      = TRUE,
  check.names = FALSE
)
head(all_taxa_2)
#Other_taxa is already added

In [ ]:
meta_data_tp2 <- read.csv(file.path(base_path, "meta_data_tp2.csv"))
meta_data_tp2 <- meta_data_tp2 %>%
  mutate(kit2.faecal_sample.barcode = as.character(kit2.faecal_sample.barcode))


In [ ]:
meta_data_tp2$Primipara <- factor(meta_data_tp2$Primipara, levels = c("0","1"))
meta_data_tp2$excess_weight <- factor(meta_data_tp2$excess_weight, levels = c("No","Yes"))
meta_data_tp2$Gest_Diabetes <- factor(meta_data_tp2$Gest_Diabetes,levels = c("0","1"))
meta_data_tp2$Q1_healthydiet <- factor(meta_data_tp2$Q1_healthydiet, levels = c("1","0"))
meta_data_tp2$Q2_healthydiet <- factor(meta_data_tp2$Q2_healthydiet, levels = c("1","0"))
meta_data_tp2$Group <- factor(meta_data_tp2$Group, levels = c("Control","Case"))


In [ ]:
beta_taxa2 <- all_taxa_2[,colnames(all_taxa_2) %in% meta_data_tp2$kit2.faecal_sample.barcode]
head(beta_taxa2)
dim(beta_taxa2)

In [ ]:
for_arrange_2 <- as.data.frame(t(beta_taxa2))
head(for_arrange_2)
dim(for_arrange_2)

In [ ]:
order_taxa_2 <- for_arrange_2[meta_data_tp2$kit2.faecal_sample.barcode, , drop = FALSE]
head(order_taxa_2)
dim(order_taxa_2)

In [ ]:
identical(rownames(order_taxa_2), meta_data_tp2$kit2.faecal_sample.barcode)

In [ ]:
jaccard_2 <- order_taxa_2 %>%                           
  decostand(method = "pa") %>%          
  vegdist(method = "jaccard")

In [ ]:
aitchison_2 <- order_taxa_2 %>%
  as.matrix() %>%
  vegdist(method = "robust.aitchison")
head(as.matrix(aitchison_2))

In [ ]:
bray_curtis_2 <- vegan::vegdist(order_taxa_2, method = "bray")
head(bray_curtis_2)

## 2. betadisper dispersion test


### TP1

In [ ]:
dispersion_J_tp1 <- vegan::betadisper(jaccard, meta_data_tp1$Group)
permutest(dispersion_J_tp1)

In [ ]:
boxplot(dispersion_J_tp1 )
plot(dispersion_J_tp1 )

In [ ]:
dispersion_tp1_At <- vegan::betadisper(aitchison , meta_data_tp1$Group)
permutest(dispersion_tp1_At)

In [ ]:
boxplot(dispersion_tp1_At)
plot(dispersion_tp1_At)

In [ ]:
dispersion_tp1_BC <- vegan::betadisper(bray_curtis, meta_data_tp1$Group)
permutest(dispersion_tp1_BC )

In [ ]:
boxplot(dispersion_tp1_BC)
plot(dispersion_tp1_BC)

### TP2

In [ ]:
dispersion_J_tp2 <- vegan::betadisper(jaccard_2, meta_data_tp2$Group)
permutest(dispersion_J_tp2)

In [ ]:
dispersion_J_tp2 <- vegan::betadisper(jaccard_2, meta_data_tp2$Group)
permutest(dispersion_J_tp2)

In [ ]:
boxplot(dispersion_J_tp2)
plot(dispersion_J_tp2)

In [ ]:
dispersion_tp2_At <- vegan::betadisper(aitchison_2 , meta_data_tp2$Group)
permutest(dispersion_tp2_At)

In [ ]:
boxplot(dispersion_tp2_At)
plot(dispersion_tp2_At)

In [ ]:
dispersion_tp2_BC <- vegan::betadisper(bray_curtis_2, meta_data_tp2$Group)
permutest(dispersion_tp2_BC )

In [ ]:
boxplot(dispersion_tp2_BC)
plot(dispersion_tp2_BC)

## 3. Random subsampling for Jaccard dispersion


In [ ]:
set.seed(123)
n_subsample <- 245
n_iterations <- 100

jaccard_betadisper_equalN <- numeric(n_iterations)
LGA_all <- meta_data_tp1 %>% filter(Group == "Case")%>% pull("kit1.faecal_sample.barcode")

AGA_all <- meta_data_tp1 %>% filter(Group == "Control") %>% pull("kit1.faecal_sample.barcode")
jaccard_mat <- as.matrix(jaccard) 

for(i in 1:n_iterations){
  
  # Keep ALL LGA — subsample AGA to n=245
  AGA_sub <- sample(AGA_all,
                    n_subsample,
                    replace = FALSE)
  sub_samples <- c(LGA_all, AGA_sub)
  
    meta_sub <-  meta_data_tp1 %>%
    filter(kit1.faecal_sample.barcode %in% sub_samples)
  
  # Subset Jaccard matrix
  jaccard_sub <- as.dist(
    jaccard_mat[sub_samples, sub_samples]
  )
  
  # Betadisper
  bd_jaccard <- betadisper(
    jaccard_sub,
    meta_sub$Group
  )
  jaccard_betadisper_equalN[i] <- permutest(
    bd_jaccard,
    permutations = 99
  )$tab$`Pr(>F)`[1]
  
  if(i %% 100 == 0) cat("Iteration", i,
                          "complete\n")
}

In [ ]:

cat("Betadisper non-significant (p>0.05):",
    mean(jaccard_betadisper_equalN > 0.05), "\n")
cat("Betadisper median p:",
    round(median(jaccard_betadisper_equalN), 3), "\n")


## 4. PERMANOVA (unadjusted and adjusted) at TP1 and TP2


### TP1

In [ ]:
set.seed(1234)
jaccard_adonis_tp1 <- vegan::adonis2(jaccard ~ Group, data = meta_data_tp1)
jaccard_adonis_tp1

In [ ]:
set.seed(1234)
jaccard_adonis_tp1_adjust <- vegan::adonis2(jaccard ~ Group + age_checked + BMI_prior + Primipara  + Q1_healthydiet, data = meta_data_tp1 , by = "margin")
jaccard_adonis_tp1_adjust 

In [ ]:
set.seed(1234)
adonis_test_group_At_tp1 <- vegan::adonis2(aitchison ~ Group, data = meta_data_tp1)
adonis_test_group_At_tp1

In [ ]:
set.seed(1234)
adonis_test_adjust_At_tp1 <- vegan::adonis2(aitchison ~ Group + age_checked + BMI_prior + Primipara  + Q1_healthydiet, data = meta_data_tp1, by = "margin")
adonis_test_adjust_At_tp1

In [ ]:
set.seed(1234)
adonis_test_group_BC_tp1 <- vegan::adonis2(bray_curtis ~ Group, data = meta_data_tp1)
adonis_test_group_BC_tp1

In [ ]:
set.seed(1234)
adonis_test_adjust_BC_tp1 <- vegan::adonis2(bray_curtis ~ Group + age_checked + BMI_prior + Primipara  + Q1_healthydiet, data = meta_data_tp1, by = "margin")
adonis_test_adjust_BC_tp1

### TP2

In [ ]:
set.seed(1234)
adonis_test_group_j_2 <- vegan::adonis2(jaccard_2 ~ Group, data = meta_data_tp2)
adonis_test_group_j_2

In [ ]:
set.seed(1234)
adonis_test_adjust_j_2 <- vegan::adonis2(jaccard_2 ~ Group + age_checked + BMI_prior + Primipara  + Q2_healthydiet, data = meta_data_tp2, by = "margin")
adonis_test_adjust_j_2

In [ ]:
set.seed(1234)
adonis_test_group_At_2<- vegan::adonis2(aitchison_2 ~ Group, data = meta_data_tp2)
adonis_test_group_At_2

In [ ]:
set.seed(1234)
adonis_test_adjust_At_2 <- vegan::adonis2(aitchison_2 ~ Group + age_checked + BMI_prior + Primipara  + Q2_healthydiet, data = meta_data_tp2, by = "margin")
adonis_test_adjust_At_2 

In [ ]:
set.seed(1234)
adonis_test_group_BC_2 <- vegan::adonis2(bray_curtis_2 ~ Group, data = meta_data_tp2)
adonis_test_group_BC_2


In [ ]:
set.seed(1234)
adonis_test_adjust_BC_2 <- vegan::adonis2(bray_curtis_2 ~ Group + age_checked + BMI_prior + Primipara  + Q2_healthydiet, data = meta_data_tp2, by = "margin")
adonis_test_adjust_BC_2

## 5. PCoA ordination plots and Bar plots (Figure 4)


### TP1

In [ ]:
jaccard_pcoa_1 <-wcmdscale(jaccard, eig = TRUE, add = TRUE)
jaccard_points_1 <- data.frame(jaccard_pcoa_1$points)
#head(jaccard_points)
jaccard_points_1 <- data.frame(jaccard_pcoa_1$points)
head (jaccard_points_1)

In [ ]:
# Merge the points with metadata
jaccard_merged_1 <- jaccard_points_1 %>%
  rownames_to_column(var = "kit1.faecal_sample.barcode") %>%
  left_join(meta_data_tp1, by = "kit1.faecal_sample.barcode")
head(jaccard_merged_1)

In [ ]:
# Save the eigenvalues, add percentages for the plot
jaccard_dim1 <- round(jaccard_pcoa_1$eig[1]/sum(jaccard_pcoa_1$eig)*100, digits = 1)
jaccard_dim2 <- round(jaccard_pcoa_1$eig[2]/sum(jaccard_pcoa_1$eig)*100,  digits = 1)

In [ ]:
aitchison_pcoa_1 <- wcmdscale(aitchison, eig = TRUE)
aitchison_points_1 <- data.frame(aitchison_pcoa_1$points)
aitchison_merged_1 <- aitchison_points_1 %>%
  rownames_to_column(var = "kit1.faecal_sample.barcode") %>%
  left_join(meta_data_tp1, by = "kit1.faecal_sample.barcode")
head(aitchison_merged_1)

In [ ]:
# Save the eigenvalues, add percentages for plot
aitchison_dim1 <- round(aitchison_pcoa_1$eig[1]/sum(aitchison_pcoa_1$eig)*100, digits = 1)
aitchison_dim2 <- round(aitchison_pcoa_1$eig[2]/sum(aitchison_pcoa_1$eig)*100,  digits = 1)
aitchison_dim1
aitchison_dim2

In [ ]:
# PCoA using cmdscale 
bray_curtis_pcoa_1 <- cmdscale(bray_curtis, eig = TRUE, k = 8) 

# Convert the result into a data frame for easy manipulation
bray_curtis_points_1 <- as.data.frame(bray_curtis_pcoa_1$points)
colnames(bray_curtis_points_1) <- paste0("X", 1:ncol(bray_curtis_points_1))  

# Merge the PCoA results with the metadata
bray_curtis_merged_1 <- bray_curtis_points_1 %>%
  rownames_to_column(var = "kit1.faecal_sample.barcode") %>%
  left_join(meta_data_tp1, by = "kit1.faecal_sample.barcode")


head(bray_curtis_merged_1)

In [ ]:
bray_curtis_pcoa_df_1 <- data.frame(kit1.faecal_sample.barcode = bray_curtis_merged_1[,1],
                                  Group = bray_curtis_merged_1$Group,
                                  pcoa1 = bray_curtis_merged_1[,2], 
                                  pcoa2 = bray_curtis_merged_1[,3])

head(bray_curtis_pcoa_df_1)

In [ ]:
# Save the eigenvalues, add percentages for the plot
bray_dim1 <- round(bray_curtis_pcoa_1$eig[1] / sum(bray_curtis_pcoa_1$eig) * 100, digits = 1)
bray_dim2 <- round(bray_curtis_pcoa_1$eig[2] / sum(bray_curtis_pcoa_1$eig) * 100, digits = 1)


bray_dim1
bray_dim2

In [ ]:
my_colours <- c("Case" = "deeppink","Control" = "#40E0D0")

In [ ]:
plot_tp1 <- ggplot(data = jaccard_merged_1,
       aes(x = Dim1, y = Dim2, color = Group)) +
  geom_point() + 
  stat_ellipse() +
  labs(title = "Jaccard distance",
       y= paste0("PCoA2 [", jaccard_dim2, "%]"), x = paste0(" PCoA1 [", jaccard_dim1, "%]")) +
  scale_color_manual(values = my_colours) +
  theme_minimal() +
  theme(axis.title.x = element_text(size = 10, face = "bold"),
                axis.text.x = element_text(size = 8, face = "bold"),
                axis.title.y = element_text(size = 10, face = "bold"),
                axis.text.y = element_text(size = 8, face = "bold"),
                plot.title = element_text(size = 12, face = "bold"))

plot_tp2 <- ggplot(data = aitchison_merged_1,
       aes(x = Dim1, y = Dim2, color = Group)) +
  geom_point() +
  stat_ellipse() +
  labs(title = "Aitchison Distance",
       y= paste0("PCoA2 [", aitchison_dim2, "%]"), x = paste0(" PCoA1 [", aitchison_dim1, "%]")) + 
  scale_color_manual(values = my_colours) +
  theme_minimal() +
  theme(axis.title.x = element_text(size = 10, face = "bold"),
                axis.text.x = element_text(size = 8, face = "bold"),
                axis.title.y = element_text(size = 10, face = "bold"),
                axis.text.y = element_text(size = 8, face = "bold"),
                plot.title = element_text(size = 12, face = "bold"))

plot_tp3 <- ggplot(data = bray_curtis_pcoa_df_1, aes(x = pcoa1, y = pcoa2, color = Group)) +
  geom_point() +
  stat_ellipse() +
  labs(title = "Bray-Curtis",
       y = paste0("PCoA2 [", bray_dim2, "%]"), 
       x = paste0(" PCoA1 [", bray_dim1, "%]")) +
  scale_color_manual(values = my_colours) + 
  theme_minimal() + 
  theme(axis.title.x = element_text(size = 10, face = "bold"),
                axis.text.x = element_text(size = 8, face = "bold"),
                axis.title.y = element_text(size = 10, face = "bold"),
                axis.text.y = element_text(size = 8, face = "bold"),
                plot.title = element_text(size = 12, face = "bold"))



In [ ]:
pcoa_tp1 <- (plot_tp1 | plot_tp2) / plot_tp3 + 
  plot_layout(guides = 'collect') &
  theme(legend.position = "right",
        legend.title = element_text(size = 14, face = "bold"),
        legend.text = element_text(size = 12))

pcoa_tp1 

In [ ]:
bc_1 <- adonis_test_adjust_BC_tp1 %>%
  mutate(metric = "Bray-Curtis",
         variable = rownames(.))

at_1 <- adonis_test_adjust_At_tp1 %>%
  mutate(metric = "Aitchison",
         variable = rownames(.))

jac_1 <- jaccard_adonis_tp1_adjust  %>%
  mutate(metric = "Jaccard",
         variable = rownames(.))

combined_PMN_1 <- bind_rows(bc_1, at_1, jac_1)
combined_PMN_1 

In [ ]:
combined_PMN_1  <- combined_PMN_1  %>%
  filter(!variable %in% c("Residual", "Total"))
combined_PMN_1

In [ ]:
combined_PMN_1$variable <- factor(combined_PMN_1$variable,
                                levels = c("Group","age_checked", "BMI_prior","Primipara", "Q1_healthydiet"))

In [ ]:
combined_PMN_1 <- combined_PMN_1 %>%
  mutate(variable = recode(variable,
                           "age_checked" = "Maternal_age",
                           "Q1_healthydiet" = "Healthy_diet",
                          "BMI_prior" = "Pre-pregnancy_BMI",
                         "Primipara" = "Primiparous"))

In [ ]:
adonis_plot_1 <- ggplot(combined_PMN_1, aes(x = variable, y = R2, fill = metric)) +
  geom_bar(stat = "identity", position = position_dodge()) +
  geom_text(aes(label = ifelse(`Pr(>F)` < 0.05, "*", "")),
            position = position_dodge(width = 0.9),
            hjust = -0.2) +  # use hjust for horizontal bars
  scale_fill_manual(
    values = c(
      "Bray-Curtis" = "#4E79A7",   # teal, #3D7A8A
      "Aitchison"   = "#F28E2B",   # coral, #E07A5F
      "Jaccard"     = "#9E9E9E"    # gray, #888780
    )
  ) +
  labs(x = "Variables", y = "R²", fill = "Distance Metric") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 0, hjust = 1)) +
  coord_flip() +
#theme(
  #legend.title = element_text(size = 15, colour = "black"),
  #legend.text = element_text(size = 13, colour = "black")
#)

theme_minimal(base_size = 16) +
theme(
  axis.text = element_text(colour = "black"),
  axis.title = element_text(colour = "black",face = "bold"),
  legend.text = element_text(colour = "black"),
  legend.title = element_text(colour = "black",face = "bold")
)
adonis_plot_1

### TP2

In [ ]:
jaccard_pcoa_2 <-wcmdscale(jaccard_2, eig = TRUE, add = TRUE)
jaccard_points_2 <- data.frame(jaccard_pcoa_2$points)
jaccard_points_2 <- data.frame(jaccard_pcoa_2$points)

In [ ]:
# Merge the points with metadata
jaccard_merged_2 <- jaccard_points_2 %>%
  rownames_to_column(var = "kit2.faecal_sample.barcode") %>%
  left_join(meta_data_tp2, by = "kit2.faecal_sample.barcode")


In [ ]:
# Save the eigenvalues, add percentages for the plot
jaccard_dim1_2 <- round(jaccard_pcoa_2$eig[1]/sum(jaccard_pcoa_2$eig)*100, digits = 1)
jaccard_dim2_2 <- round(jaccard_pcoa_2$eig[2]/sum(jaccard_pcoa_2$eig)*100,  digits = 1)

In [ ]:
aitchison_pcoa_2 <- wcmdscale(aitchison_2, eig = TRUE)
aitchison_points_2 <- data.frame(aitchison_pcoa_2$points)
# Merge the PCoA results with the metadata
aitchison_merged_2 <- aitchison_points_2 %>%
  rownames_to_column(var = "kit2.faecal_sample.barcode") %>%
  left_join(meta_data_tp2, by = "kit2.faecal_sample.barcode")

In [ ]:
# Save the eigenvalues, add percentages for plot
aitchison_dim11 <- round(aitchison_pcoa_2$eig[1]/sum(aitchison_pcoa_2$eig)*100, digits = 1)
aitchison_dim22 <- round(aitchison_pcoa_2$eig[2]/sum(aitchison_pcoa_2$eig)*100,  digits = 1)

In [ ]:
bray_curtis_pcoa_2 <- cmdscale(bray_curtis_2, eig = TRUE, k = 8)
# Convert the result into a data frame for easy manipulation
bray_curtis_points_2 <- as.data.frame(bray_curtis_pcoa_2$points)
colnames(bray_curtis_points_2) <- paste0("X", 1:ncol(bray_curtis_points_2))


In [ ]:
# Merge the PCoA results with the metadata
bray_curtis_merged_2 <- bray_curtis_points_2 %>%
  rownames_to_column(var = "kit2.faecal_sample.barcode") %>%
  left_join(meta_data_tp2, by = "kit2.faecal_sample.barcode")


In [ ]:
bray_curtis_pcoa_df_2 <- data.frame(kit2.faecal_sample.barcode = bray_curtis_merged_2[,1],
                                  Group = bray_curtis_merged_2$Group,
                                  pcoa1 = bray_curtis_merged_2[,2], 
                                  pcoa2 = bray_curtis_merged_2[,3])

In [ ]:
# Save the eigenvalues, add percentages for the plot
bray_dim11 <- round(bray_curtis_pcoa_2$eig[1] / sum(bray_curtis_pcoa_2$eig) * 100, digits = 1)
bray_dim22 <- round(bray_curtis_pcoa_2$eig[2] / sum(bray_curtis_pcoa_2$eig) * 100, digits = 1)

In [ ]:
my_colours_2 <- c("Case" = "#E30B5D","Control" = "#708090")

In [ ]:
plot_11 <- ggplot(data = jaccard_merged_2,
       aes(x = Dim1, y = Dim2, color = Group)) +
  geom_point() + 
  stat_ellipse() +
  labs(title = "Jaccard distance",
       y= paste0("PCoA2 [", jaccard_dim2_2, "%]"), x = paste0(" PCoA1 [", jaccard_dim1_2, "%]")) +
  scale_color_manual(values = my_colours_2) +
  theme_minimal() +
  theme(axis.title.x = element_text(size = 10, face = "bold"),
                axis.text.x = element_text(size = 8, face = "bold"),
                axis.title.y = element_text(size = 10, face = "bold"),
                axis.text.y = element_text(size = 8, face = "bold"),
                plot.title = element_text(size = 12, face = "bold"))

plot_22 <- ggplot(data = aitchison_merged_2,
       aes(x = Dim1, y = Dim2, color = Group)) +
  geom_point() +
  stat_ellipse() +
  labs(title = "Aitchison Distance",
       y= paste0("PCoA2 [", aitchison_dim22, "%]"), x = paste0(" PCoA1 [", aitchison_dim11, "%]")) + 
  scale_color_manual(values = my_colours_2) +
  theme_minimal() +
  theme(axis.title.x = element_text(size = 10, face = "bold"),
                axis.text.x = element_text(size = 8, face = "bold"),
                axis.title.y = element_text(size = 10, face = "bold"),
                axis.text.y = element_text(size = 8, face = "bold"),
                plot.title = element_text(size = 12, face = "bold"))

plot_33 <- ggplot(data = bray_curtis_pcoa_df_2, aes(x = pcoa1, y = pcoa2, color = Group)) +
  geom_point() +
  stat_ellipse() +
  labs(title = "Bray-Curtis",
       y = paste0("PCoA2 [", bray_dim22, "%]"), 
       x = paste0(" PCoA1 [", bray_dim11, "%]")) +
  scale_color_manual(values = my_colours_2) + 
  theme_minimal() + 

  theme(axis.title.x = element_text(size = 10, face = "bold"),
                axis.text.x = element_text(size = 8, face = "bold"),
                axis.title.y = element_text(size = 10, face = "bold"),
                axis.text.y = element_text(size = 8, face = "bold"),
                plot.title = element_text(size = 12, face = "bold"))


pcoa_tp2 <- (plot_11 | plot_22) / plot_33 + 
  plot_layout(guides = 'collect') &
  theme(legend.position = "right",
        legend.title = element_text(size = 14, face = "bold"),
        legend.text = element_text(size = 12))

pcoa_tp2

In [ ]:
bc_2 <- adonis_test_adjust_BC_2 %>%
  mutate(metric = "Bray-Curtis",
         variable = rownames(.))

at_2 <- adonis_test_adjust_At_2  %>%
  mutate(metric = "Aitchison",
         variable = rownames(.))

jac_2 <- adonis_test_adjust_j_2 %>%
  mutate(metric = "Jaccard",
         variable = rownames(.))

In [ ]:
combined_PMN_2 <- bind_rows(bc_2, at_2, jac_2)
combined_PMN_2  <- combined_PMN_2  %>%
  filter(!variable %in% c("Residual", "Total"))
combined_PMN_2$variable <- factor(combined_PMN_2$variable,
                                levels = c("Group","age_checked", "BMI_prior","Primipara", "Q2_healthydiet"))

combined_PMN_2 <- combined_PMN_2 %>%
  mutate(variable = recode(variable,
                           "age_checked" = "Maternal_age",
                           "Q2_healthydiet" = "Healthy_diet",
                          "BMI_prior" = "Pre-pregnancy_BMI",
                         "Primipara" = "Primiparous"))



In [ ]:
adonis_plot_2 <- ggplot(combined_PMN_2, aes(x = variable, y = R2, fill = metric)) +
  geom_bar(stat = "identity", position = position_dodge()) +
  geom_text(aes(label = ifelse(`Pr(>F)` < 0.05, "*", "")),
            position = position_dodge(width = 0.9),
            hjust = -0.2) +  # use hjust for horizontal bars
  scale_fill_manual(values = c("Bray-Curtis" = "#4E79A7",
                               "Aitchison" = "#F28E2B",
                               "Jaccard" = "#9E9E9E")) +
  labs(x = "Variables", y = "R²", fill = "Distance Metric") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 0, hjust = 1)) +
  coord_flip() + # flips to horizontal bars
    theme_minimal(base_size = 16) +
    theme(
  axis.text = element_text(colour = "black"),
  axis.title = element_text(colour = "black",face = "bold"),
  legend.text = element_text(colour = "black"),
  legend.title = element_text(colour = "black",face = "bold")
) 

adonis_plot_2

### Figure 4

In [ ]:
final_figure_beta<-
  (wrap_elements(pcoa_tp1) + ggtitle("A) Beta Diversity at TP1") |
   wrap_elements(adonis_plot_1)   + ggtitle("B) Bar plot of variance explained at TP1")) /
  (wrap_elements(pcoa_tp2) + ggtitle("C) Beta Diversity at TP2") |
   wrap_elements(adonis_plot_2)    + ggtitle("D) Bar plot of variance explained at TP2")) +
  plot_layout(heights = c(1, 1)) &
  theme(plot.title = element_text(size = 14, face = "bold", hjust = 0))

final_figure_beta

In [ ]:
#pdf
ggsave("figure_4.pdf",
       final_figure_beta,
       width     = 380,     
       height    = 260,
       units     = "mm",
       limitsize = FALSE,
       device    = cairo_pdf)

In [ ]:
# TIFF 
ggsave("figure_4.tiff",
       final_figure_beta,
       width       = 380,
       height      = 260,
       units       = "mm",
       limitsize   = FALSE,
       device      = "tiff",
       dpi         = 300,
       compression = "lzw")

# EPS 
ggsave("figure_4.eps",
       final_figure_beta,
       width     = 380,
       height    = 260,
       units     = "mm",
       limitsize = FALSE,
       device    = cairo_ps)